# `scheme_name` 03: related-feature analysis

            **Purpose:** identify predictors that represent the same concept, form a
            hierarchy, share a missingness process or plausibly interact with
            `scheme_name`.

            ## Relationships selected in advance

            - `scheme_management` — Scheme names map imperfectly to the management label.
- `management` — Management gives a complete lower-cardinality alternative.
- `funder` — Funders may repeatedly support named schemes.
- `installer` — Installers may repeatedly construct named schemes.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_stage_directory():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "data" / "TrainingSetValues.csv").exists()
            and (candidate / "src" / "source_data_validation.py").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the stage-1-pump-it-up directory.")


stage_directory = find_stage_directory()
source_directory = str((stage_directory / "src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    numeric_summary,
    numeric_target_summary,
    related_feature_summary,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = stage_directory / "data"
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)

feature = 'scheme_name'
feature_metadata = {'order': 21, 'name': 'scheme_name', 'audit_type': 'high-cardinality-category', 'role': 'candidate', 'disposition': 'exclude raw one-hot value from the first baseline', 'finding': 'Nearly half the source values are blank and the recorded names are high-cardinality and inconsistent with scheme management.', 'decision': 'Preserve blank and literal sentinels separately; test only fold-fitted high-cardinality treatments.', 'risk': 'Scheme identity can memorise projects and geography.', 'sentinel_tokens': ['None', 'none', 'no scheme', 'not known'], 'related': [{'feature': 'scheme_management', 'reason': 'Scheme names map imperfectly to the management label.'}, {'feature': 'management', 'reason': 'Management gives a complete lower-cardinality alternative.'}, {'feature': 'funder', 'reason': 'Funders may repeatedly support named schemes.'}, {'feature': 'installer', 'reason': 'Installers may repeatedly construct named schemes.'}]}
feature_types = {'amount_tsh': 'numeric', 'date_recorded': 'date', 'funder': 'high-cardinality-category', 'gps_height': 'numeric', 'installer': 'high-cardinality-category', 'longitude': 'coordinate', 'latitude': 'coordinate', 'wpt_name': 'high-cardinality-category', 'num_private': 'numeric', 'basin': 'category', 'subvillage': 'high-cardinality-category', 'region': 'category', 'region_code': 'category', 'district_code': 'category', 'lga': 'category', 'ward': 'high-cardinality-category', 'population': 'numeric', 'public_meeting': 'binary', 'recorded_by': 'constant', 'scheme_management': 'category', 'scheme_name': 'high-cardinality-category', 'permit': 'binary', 'construction_year': 'year', 'extraction_type': 'category', 'extraction_type_group': 'category', 'extraction_type_class': 'category', 'management': 'category', 'management_group': 'category', 'payment': 'category', 'payment_type': 'category', 'water_quality': 'category', 'quality_group': 'category', 'quantity': 'category', 'quantity_group': 'category', 'source': 'category', 'source_type': 'category', 'source_class': 'category', 'waterpoint_type': 'category', 'waterpoint_type_group': 'category'}
assert feature in training_features.columns
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {feature}."
)


Validated 59,400 training rows and 14,850 test rows for scheme_name.


In [2]:
relationship_inventory = pd.DataFrame(feature_metadata["related"])
display(relationship_inventory)

relationship_evidence = related_feature_summary(
    training_features,
    feature,
    feature_metadata["audit_type"],
    feature_metadata["related"],
    feature_types,
)
display(relationship_evidence)


,feature,reason
0,scheme_management,Scheme names map imperfectly to the management...
1,management,Management gives a complete lower-cardinality ...
2,funder,Funders may repeatedly support named schemes.
3,installer,Installers may repeatedly construct named sche...


,primary,related,measure,association,complete rows,primary levels,related levels,forward modal purity (%),reverse modal purity (%),relationship rationale
0,scheme_name,scheme_management,bias-corrected Cramer's V,0.5347,59400,2527,13,75.94,47.81,Scheme names map imperfectly to the management...
1,scheme_name,management,bias-corrected Cramer's V,0.5498,59400,2527,12,78.88,47.45,Management gives a complete lower-cardinality ...
2,scheme_name,funder,bias-corrected Cramer's V,0.3546,59400,2527,1898,47.73,56.75,Funders may repeatedly support named schemes.
3,scheme_name,installer,bias-corrected Cramer's V,0.3461,59400,2527,1919,59.15,57.56,Installers may repeatedly construct named sche...


## Discussion and modelling consequence

The relationships above were nominated before inspecting the pairwise
coefficients. A strong association can mean useful interaction, hierarchy,
shared collection behaviour or redundancy; it is not a reason to keep both
fields automatically.

For `scheme_name`, carry the relationships into controlled ablations
and fit every learned grouping or encoding inside the training fold. The
current provisional disposition remains: **exclude raw one-hot value from the first baseline**.
